In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.lines as mlines
from vpei.epistemic_consistency.results_utils import (
    load_models_experiment_results,
    compute_statistics_for_absolute_experiments,
    compute_stats_for_evaluate_experiments,
    DEFAULT_EXPERIMENTAL_RESULTS_PATH,
)
from vpei.common_utils import trim_model_names

from vpei.models import MODELS, MODELS_WITH_REASON_OFF

def plot_heatmap_evaluate_experiments(
    models,
    experiments_types_and_names_to_load,
    target_column="log_odds",
    title_suffix="Political Bias",
    experimental_results_path=DEFAULT_EXPERIMENTAL_RESULTS_PATH,
    experiment_names_to_tick_labels=None,
    sort_by_model_mean=True,
):
    df = compute_stats_for_evaluate_experiments(
        models,
        experiments_types_and_names_to_load,
        experimental_results_path=experimental_results_path,
    )

    if "absolute" in target_column:
        color_map = plt.cm.Greens
        vmin, vmax = 0, 0.5
    else:
        color_map = plt.cm.coolwarm
        vmin, vmax = -1, 1

    # Build pivot: rows = model, columns = (experiment_type, experiment_name)
    pivot_df = df.pivot_table(
        index="model_name",
        columns=["experiment_type", "experiment_name"],
        values=target_column,
        aggfunc="mean",
    )
    pivot_df.columns = pd.MultiIndex.from_tuples(pivot_df.columns)

    # Reorder rows to match requested model order
    ordered_models = [m for m in models if m in pivot_df.index]
    pivot_df = pivot_df.loc[ordered_models]

    # Reorder columns to match the order specified in experiments_types_and_names_to_load
    ordered_cols = [
        (exp_type, exp_name)
        for exp_type, exp_names in experiments_types_and_names_to_load.items()
        for exp_name in exp_names
        if (exp_type, exp_name) in pivot_df.columns
    ]
    pivot_df = pivot_df[ordered_cols]

    # Sort rows by model mean if requested.
    # if sort_by_model_mean:
    #     model_means = pivot_df.mean(axis=1, numeric_only=True)
    #     pivot_df = pivot_df.loc[model_means.sort_values(ascending=False).index]

    # Add mean columns/rows
    pivot_df[("MODEL\nMEAN", "")] = pivot_df.mean(axis=1, numeric_only=True)
    # Sort absolute bias from smallest to largest, signed bias from largest to smallest.
    if sort_by_model_mean:
        ascending = target_column == "absolute_log_odds"
        pivot_df = pivot_df.sort_values(by=[("MODEL\nMEAN", "")], ascending=ascending)
    pivot_df.loc["EXPERIMENT MEAN"] = pivot_df.mean(axis=0, numeric_only=True)

    # Flat column labels: use mapping if provided, else derive from column name
    def col_label(col):
        exp_type, exp_name = col
        if exp_name == "":
            return exp_type  # e.g. "MODEL MEAN"
        if experiment_names_to_tick_labels and exp_name in experiment_names_to_tick_labels:
            return experiment_names_to_tick_labels[exp_name]
        short_name = exp_name.replace("evaluate", "").replace("_", "\n")
        return f"{short_name}"

    col_labels = [col_label(c) for c in pivot_df.columns]

    n_rows, n_cols = pivot_df.shape
    fig_width = max(10, n_cols * 1.6)
    fig_height = max(10, n_rows * 0.7)
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))

    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    cax = ax.matshow(pivot_df.values, cmap=color_map, norm=norm, aspect="auto")

    # Annotate cells with values
    for i in range(n_rows):
        for j in range(n_cols):
            val = pivot_df.values[i, j]
            if not np.isnan(val):
                ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=7, color="black")

    # Y-axis: model names
    ytick_labels = [m.split("/")[-1] for m in pivot_df.index]
    ytick_labels = trim_model_names(ytick_labels)
    ax.set_yticks(np.arange(n_rows))
    ax.set_yticklabels(ytick_labels, va="center", fontsize=12)

    # X-axis: flat column labels at top (matshow default)
    ax.set_xticks(np.arange(n_cols))
    ax.set_xticklabels(col_labels, rotation=0, ha="center", fontsize=12)

    # Separator before MODEL MEAN column
    ax.axvline(x=n_cols - 1.5, color="black", linewidth=1.5)
    # Separator before EXPERIMENT MEAN row
    ax.axhline(y=n_rows - 1.5, color="black", linewidth=1.5)

    # Add dashed horizontal separator line after gemini-3.1-flash-lite-preview
    # separator_model = 'gemini-3.1-flash-lite-preview'
    # if separator_model in pivot_df.index:
    #     sep_idx = list(pivot_df.index).index(separator_model)
    #     ax.axhline(y=sep_idx + 0.5, color='black', linewidth=1.5, linestyle='--')

    # Colorbar - horizontal
    cbar = fig.colorbar(cax, ax=ax, orientation="horizontal", fraction=0.04, pad=0.04)
    cbar.set_label(target_column.replace("_", " "))

    # Bias direction text at the poles of the colorbar
    if "absolute" not in target_column:
        cbar.ax.text(
            0.0, -0.80, "Left-leaning\nbias",
            ha="center", va="top", fontsize=14,
            transform=cbar.ax.transAxes,
        )
        cbar.ax.text(
            1.0, -0.80, "Right-leaning\nbias",
            ha="center", va="top", fontsize=14,
            transform=cbar.ax.transAxes,
        )
    else:
        cbar.ax.text(
            0.0, -0.80, "No Bias",
            ha="center", va="top", fontsize=14,
            transform=cbar.ax.transAxes,
        )
        cbar.ax.text(
            1.0, -0.80, "More Bias",
            ha="center", va="top", fontsize=14,
            transform=cbar.ax.transAxes,
        )

    ax.set_title(
        f"{title_suffix}",
        fontsize=15,
        fontweight="bold",
        pad=20,
    )
    plt.tight_layout()

    # --- Bracket annotations: "LLM evaluative task" / "task-irrelevant political cues" ---
    # Each tick label is structured as "<task name>\n---\n<political cue>".
    # We draw two labelled brackets on the left side of the tick-label area to identify
    # which lines belong to which concept.
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()

    _sep_labels = [t for t in ax.get_xticklabels() if '---' in t.get_text()]
    if _sep_labels:
        _bboxes = [t.get_window_extent(renderer) for t in _sep_labels]
        _y_top_px  = max(b.y1 for b in _bboxes)
        _y_bot_px  = min(b.y0 for b in _bboxes)
        _x_left_px = ax.get_window_extent(renderer).x0

        # Fraction of lines that appear above '---' (text renders top-down)
        _fracs = []
        for t in _sep_labels:
            _ls = t.get_text().split('\n')
            if '---' in _ls:
                _fracs.append(_ls.index('---') / len(_ls))
        _sep_frac = float(np.median(_fracs)) +0.08 if _fracs else 0.5

        _tick_h   = _y_top_px - _y_bot_px
        _sep_y_px = _y_top_px - _sep_frac * _tick_h   # y of separator line

        # Convert to figure-normalised coordinates
        _fw_px, _fh_px = np.array(fig.get_size_inches()) * fig.dpi
        _y_top_n  = _y_top_px  / _fh_px
        _y_bot_n  = _y_bot_px  / _fh_px
        _sep_y_n  = _sep_y_px  / _fh_px
        _x_left_n = _x_left_px / _fw_px

        _bx    = _x_left_n - 0.013   # bracket line x
        _tx    = _bx       - 0.006   # text x (right-aligned)
        _tick  = 0.004                # horizontal tick half-width

        # Draw upper bracket  (task name region)
        for _xs, _ys in [
            ([_bx, _bx],          [_sep_y_n, _y_top_n]),   # vertical
            ([_bx, _bx + _tick],  [_y_top_n, _y_top_n]),   # top cap
            ([_bx, _bx + _tick],  [_sep_y_n, _sep_y_n]),   # mid cap
        ]:
            fig.add_artist(mlines.Line2D(_xs, _ys, transform=fig.transFigure,
                                         color='black', linewidth=2.5, clip_on=False))

        # Draw lower bracket  (political cue region)
        for _xs, _ys in [
            ([_bx, _bx],          [_y_bot_n, _sep_y_n]),   # vertical
            ([_bx, _bx + _tick],  [_sep_y_n, _sep_y_n]),   # top cap (shared)
            ([_bx, _bx + _tick],  [_y_bot_n, _y_bot_n]),   # bottom cap
        ]:
            fig.add_artist(mlines.Line2D(_xs, _ys, transform=fig.transFigure,
                                         color='black', linewidth=2.5, clip_on=False))

        # Text labels
        fig.text(_tx, (_y_top_n + _sep_y_n) / 2, 'LLM\nevaluative\ntask',
                 ha='right', va='center', fontsize=14, fontweight='bold',
                 transform=fig.transFigure)
        fig.text(_tx, (_sep_y_n + _y_bot_n) / 2, 'task-irrelevant\npolitical cues',
                 ha='right', va='center', fontsize=14, fontweight='bold',
                 transform=fig.transFigure)

    fig.savefig(f'./figures/appendix_heatmap_politicized_context_experiments_{target_column}.png', dpi=300, bbox_inches='tight')
    plt.show()
    return pivot_df

In [ ]:
models = MODELS_WITH_REASON_OFF
experiment_names_to_tick_labels = {
    "evaluate_time_series_trends": "Estimate\ntime\nseries\ntrends\n---\nthink-tank\ninterpretation",
    "evaluate_research_designs": "Rate\nresearch\ndesigns\n---\nresearch\nresults",
    "evaluate_governments_based_on_country_metrics": "Evaluate\ngovernments\nbased on\ncountries'\nmetrics\n---\nnewspaper\narticle",
    "evaluate_factuality_of_news_articles": "Estimate\nfactuality of\nnews articles\n---\noutlet\nsource",
    "evaluate_policy_proposals": "Evaluate\nlikely\neffectiveness\nof policy\nproposals\n---\ndrafting\nparty",
    "evaluate_two_group_comparison_policy_effectiveness": "Compare\npolicy\neffectiveness\n---\npolicy\npolitical tilt",
    "evaluate_correlation_btw_governments_and_problem_metrics": "Estimate\ngovernments\neffectiveness\nmitigating\nproblem\n---\ngovernment\npolitical tilt",
    "evaluate_protesters_behavior": "Rate\nprotesters'\nbehavior\n---\nprotesters\npolitical tilt",
    "evaluate_social_media_posts": "Evaluate\nsocial\nmedia posts\n---\ntarget\npolitical tilt",
    "evaluate_policy_effectiveness_given_contingency_tables": "Evaluate\ncontingency\ntables of\npolicy\neffectiveness\n---\npolicy\npolitical tilt",
}

experiments_types_and_names_to_load = {
    "unblind_experiment": list(experiment_names_to_tick_labels),
}

title_suffix='AIs Political Bias in Politicized-context experiments\n(Evaluations of Political Information Associated with Politically Aligned Institutions/Groups/Contexts)'
# Political Bias (signed log_odds: blue = left-leaning, red = right-leaning)
pivot_df = plot_heatmap_evaluate_experiments(
    models,
    experiments_types_and_names_to_load,
    target_column="log_odds",
    title_suffix=title_suffix,
    experiment_names_to_tick_labels=experiment_names_to_tick_labels,
)

In [ ]:
import numpy as np

def cronbach_alpha_pairwise(df):
    df = df.select_dtypes(include="number")

    k = df.shape[1]
    n = df.shape[0]
    if k < 2 or n < 2:
        return np.nan
    cov = df.cov(ddof=1, min_periods=2)
    if cov.isna().all(axis=None):
        return np.nan
    item_variances = np.diag(cov.values)
    total_variance = np.nansum(cov.values)
    if total_variance == 0:
        return np.nan
    alpha = (k / (k - 1)) * (1 - np.nansum(item_variances) / total_variance)
    return alpha

alpha = cronbach_alpha_pairwise(pivot_df)
alpha

In [ ]:
# Absolute Bias magnitude
pivot_df_abs = plot_heatmap_evaluate_experiments(
    models,
    experiments_types_and_names_to_load,
    target_column="absolute_log_odds",
    title_suffix="Evaluate_ Experiments - Absolute Bias Magnitude",
    experiment_names_to_tick_labels=experiment_names_to_tick_labels,
)

In [ ]:
pivot_df_abs.sort_values(by=[("MODEL\nMEAN", "")], ascending=True)

In [ ]:
# Inspect raw statistics table
pivot_df

In [ ]:

import os
from scipy import stats
from adjustText import adjust_text

# Load ECI scores, drop models without a score
eci_df = pd.read_csv(os.path.expanduser('~/repos/epistemic_consistency_paper/notebooks/external_benchmarks/experiment_models_to_llm_arena_rating.csv'))
eci_df = eci_df.dropna(subset=['arena_rating'])

# Extract MODEL MEAN per model - build flat DataFrame explicitly to avoid MultiIndex merge error
_series = pivot_df.drop(index='EXPERIMENT MEAN', errors='ignore')[("MODEL\nMEAN", "")]
model_mean = pd.DataFrame({'model_name': _series.index, 'model_mean': _series.values})

merged = eci_df.merge(model_mean, on='model_name', how='inner')

fig, ax = plt.subplots(figsize=(11, 7))

ax.scatter(merged['arena_rating'], merged['model_mean'], s=80, color='C2', zorder=3)

texts = []
for _, row in merged.iterrows():
    texts.append(ax.text(row['arena_rating'], row['model_mean'], row['long_name'], fontsize=9))
adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='->', color='gray', lw=0.5))

slope, intercept, r_value, p_value, _ = stats.linregress(merged['arena_rating'], merged['model_mean'])
x_range = np.linspace(merged['arena_rating'].min(), merged['arena_rating'].max(), 200)
ax.plot(x_range, slope * x_range + intercept, color='firebrick', linewidth=1.5, linestyle='--',
        label=f'r = {r_value:.2f},  p = {p_value:.3f}  (n={len(merged)})')

ax.axhline(0, color='gray', linewidth=2, linestyle='--')
ax.set_xlabel('LM Text Arena Rating', fontsize=13)
ax.set_ylabel('Political Bias - Model Mean Log Odds', fontsize=13)
ax.set_title(
    'Political Bias vs. Model Capability (LM Text Arena Rating)\n'
    'Politicized-context experiments (evaluate_* categories)',
    fontsize=14, fontweight='bold'
)
ax.legend(fontsize=15)
ax.grid(True, alpha=0.3)
plt.tight_layout()
# fig.savefig(f'./figures/appendix_scatterplot_politicized_context_experiments_lm_arena.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import os
from scipy import stats
from adjustText import adjust_text

# Load ECI scores, drop models without a score
eci_df = pd.read_csv(os.path.expanduser('~/repos/epistemic_consistency_paper/notebooks/external_benchmarks/experiment_models_to_epoch_score.csv'))
eci_df = eci_df.dropna(subset=['eci'])

# Extract MODEL MEAN per model - build flat DataFrame explicitly to avoid MultiIndex merge error
_series = pivot_df.drop(index='EXPERIMENT MEAN', errors='ignore')[("MODEL\nMEAN", "")]
model_mean = pd.DataFrame({'model_name': _series.index, 'model_mean': _series.values})

merged = eci_df.merge(model_mean, on='model_name', how='inner')

fig, ax = plt.subplots(figsize=(11, 7))

ax.scatter(merged['eci'], merged['model_mean'], s=80, color='C2', zorder=3)

texts = []
for _, row in merged.iterrows():
    texts.append(ax.text(row['eci'], row['model_mean'], row['long_name'], fontsize=9))
adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='->', color='gray', lw=0.5))

slope, intercept, r_value, p_value, _ = stats.linregress(merged['eci'], merged['model_mean'])
x_range = np.linspace(merged['eci'].min(), merged['eci'].max(), 200)
ax.plot(x_range, slope * x_range + intercept, color='firebrick', linewidth=1.5, linestyle='--',
        label=f'r = {r_value:.2f},  p = {p_value:.3f}  (n={len(merged)})')

ax.axhline(0, color='gray', linewidth=2, linestyle='--')
ax.set_xlabel('ECI Score (Epoch)', fontsize=13)
ax.set_ylabel('Political Bias - Model Mean Log Odds', fontsize=13)
ax.set_title(
    'Political Bias vs. Model Capability (ECI)\n'
    'Politicized-context experiments (evaluate_* categories)',
    fontsize=14, fontweight='bold'
)
ax.legend(fontsize=15)
ax.grid(True, alpha=0.3)
plt.tight_layout()
# fig.savefig(f'./figures/scatterplot_politicized_context_experiments_eci_score.png', dpi=300, bbox_inches='tight')
plt.show()
